# Sdlicit usability demo — JabRef ADR replay

This notebook is a small, self-contained usability demo: it feeds a real
project brief through Sdlicit's actual pipeline (Statement of Work -> SRS ->
Architecture Decision Records) and shows the generated ADRs next to
**real, historical decisions** made by the [JabRef](https://github.com/JabRef/jabref)
open-source project — an established reference manager with 50+ ADRs on GitHub.

The point isn't a benchmark score. It's simple: *here's a real architectural
decision a real engineering team made; here's what Sdlicit produces from the
same brief and topic, cold.*

**What this needs:**
- The `sdlicit` package installed (`uv sync --group examples` from `sdlicit-public/`)
- An LLM provider configured — `OPENROUTER_API_KEY` in `.env` (default), or a
  running Ollama instance (`provider: ollama` in the generated config)
- No LightRAG / knowledge-base setup required — this demo runs with
  `enable_rag=False` by default so it works out of the box. Set
  `enable_rag=True` (and pass a `kb_workdir`) to ground generation against a
  real knowledge base, the same way the CLI/extension do.

Everything this notebook does, the CLI (`cli/cli_client.py`) and the VS Code
extension do too, over the same REST API — this is just the fastest way to
see it work.

In [1]:
import json
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Markdown, display

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
load_dotenv()  # walks up from cwd - finds sdlicit-public/.env (or a parent .env in a dev checkout)

import os  # noqa: E402

from replay_pipeline import run_replay  # noqa: E402

if not os.environ.get("OPENROUTER_API_KEY") and not os.environ.get("OLLAMA_HOST"):
    print(
        "! No OPENROUTER_API_KEY (or OLLAMA_HOST) found in the environment.\n"
        "  Copy sdlicit-public/.env.example to sdlicit-public/.env and set your key\n"
        "  before running the pipeline cell below."
    )

## 1. Load the brief and pick a few real ADRs

`data/replay_dataset.jsonl` has 50+ real ADRs mined straight from JabRef's
`docs/decisions/` on GitHub — id, title, context, the options they
considered, and the decision they made. We pick a handful spanning different
domains (tooling, testing, i18n, AI/ML) so the demo doesn't just get lucky on
one easy case, and stays quick to run.

Feel free to change `ADR_IDS` to explore others — print `dataset` to see the
full list of 50+ available topics.

In [2]:
BRIEF = (HERE / "data" / "brief.md").read_text(encoding="utf-8")

dataset = {}
with open(HERE / "data" / "replay_dataset.jsonl", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        dataset[row["adr_id"]] = row

# A small, domain-diverse sample. Swap these for any of the 50+ ids in `dataset`.
ADR_IDS = ["ADR-0001", "ADR-0003", "ADR-0009", "ADR-0037"]
gold = [dataset[i] for i in ADR_IDS]
topics = [g["title"] for g in gold]

for g in gold:
    print(f"{g['adr_id']}: {g['title']}")

ADR-0001: Use Crowdin for translations
ADR-0003: Use Gradle as build tool
ADR-0009: Use Plain JUnit5 for advanced test assertions
ADR-0037: RAG Architecture Implementation


## 2. Run the Sdlicit pipeline

`run_replay` drives the real `Orchestrator` used by the FastAPI backend: it
generates a SOW from the brief, an SRS from the SOW, then walks the ADR
topics one by one — each new ADR sees every prior one as context, the same
way a human analyst builds up a decision log over a project.

This calls the configured LLM provider once per stage plus twice per ADR
(direction-suggestion + generation) — with 4 topics that's ~10 calls. Results
land in `./workspace/` if you want to inspect the raw artifacts afterwards.

In [3]:
workspace = HERE / "workspace"

result = await run_replay(
    client_brief=BRIEF,
    workspace=workspace,
    adr_topics=topics,
    provider="openrouter",
    model="openai/gpt-5.4-nano",
    enable_rag=False,   # no KB ingestion needed for this demo
    enable_socratic=False,
    enable_tom=False,
)

14:56:03 I [AGENT] KnowledgeBase DISABLED by config (enable_rag=False)


14:56:03 I [AGENT] ToMAgent DISABLED by config


14:56:03 I [AGENT] SocraticAgent DISABLED by config


14:56:03 I [AGENT] SOWAgent initialised


14:56:03 I [AGENT] ADRAgent initialised


...

14:57:55 I [AGENT] [ADRAgent] suggest_directions → 7 direction(s)


14:57:55 I [TOOL] predict  signature=ADRGeneration  inputs=['topic', 'requirements', 'prior_adrs', 'kb_context']  model=openrouter/openai/gpt-5.4-nano


  Suggestions: 7 | Target: MISS


14:58:36 I [TOOL] predict  signature=ADRGeneration → done


  Generated: 6136 chars, 5 alternatives (41.01s)

[replay] Done: 4/4 ADRs | 153.06s | suggestion hit rate: 0%


## 3. Generated vs. real

Side by side: what Sdlicit produced for each topic, against the decision
JabRef's own maintainers actually recorded.

In [4]:
def render_comparison(step, gold_row):
    gen_alts = "\n".join(f"- {a}" for a in step.alternatives) or "*(none extracted)*"
    gold_opts = gold_row.get("considered_options") or []
    gold_alts = "\n".join(f"- {o}" for o in gold_opts) or "*(none recorded)*"

    return f"""
### {gold_row['adr_id']} — {step.topic}

| | Sdlicit (generated, as "{step.adr_id}") | JabRef (real) |
|---|---|---|
| **Alternatives considered** | {gen_alts} | {gold_alts} |
| **Decision** | {step.decision or "*(no output)*"} | {gold_row.get("decision_outcome", "")[:600]} |

*Author on record: {gold_row.get("introduced_at", {}).get("author", "?")} — [{gold_row.get("file_path", "")}](https://github.com/JabRef/jabref/blob/main/{gold_row.get("file_path", "")})*
"""

for step, gold_row in zip(result.steps, gold, strict=False):
    display(Markdown(render_comparison(step, gold_row)))

print(f"\n{result.completed_count}/{len(result.steps)} ADRs generated, "
      f"{result.total_tokens} tokens, {result.total_latency_s}s total")


### ADR-0001 — Use Crowdin for translations

| | Sdlicit (generated, as "ADR-0001") | JabRef (real) |
|---|---|---|
| **Alternatives considered** | - Local-only translation files maintained in the repository
- Use another translation management platform (e.g., Lokalise, Weblate)
- Community-sourced translation workflow without a dedicated TMS
- Custom in-house translation workflow and tooling | - Use [Crowdin](http://crowdin.com/)
- Use [popeye](https://github.com/JabRef/popeye)
- Use [Lingohub](https://lingohub.com/)
- Keep current GitHub flow. See the [Step-by-step guide](https://docs.jabref.org/contributing/how-to-translate-the-ui). |
| **Decision** | Use Crowdin as the translation management system. Manage localization assets (source strings, metadata, translation memory, glossaries, and review/approval flows) in Crowdin and integrate its delivered translation files into the application build/deployment pipeline. Ensure that the application uses the bundled translation artifacts at runtime (so translation availability does not depend on network access). External network lookup consent requirements for reference enrichment (REQ-JABREF-15) are not directly applicable to translation delivery, but any runtime/online translation retrieval must be avoided or explicitly controlled. | Chosen option: "Use Crowdin", because Crowdin is easy to use, integrates in our GitHub workflow, and is free for OSS projects. |

*Author on record: Oliver Kopp — [docs/decisions/0001-use-crowdin-for-translations.md](https://github.com/JabRef/jabref/blob/main/docs/decisions/0001-use-crowdin-for-translations.md)*



### ADR-0003 — Use Gradle as build tool

| | Sdlicit (generated, as "ADR-0002") | JabRef (real) |
|---|---|---|
| **Alternatives considered** | - Maven as the build tool
- Bazel as the build tool
- Make/CMake with custom scripts
- Other Gradle-like build systems (e.g., SBT for JVM projects) | - [Maven](https://maven.apache.org/)
- [Gradle](https://gradle.org/)
- [Ant](https://ant.apache.org/) |
| **Decision** | Adopt Gradle as the build tool for the project. Configure Gradle to support: (1) reproducible builds with pinned/locked dependency versions where applicable, (2) packaging of all runtime-required assets into build outputs (so offline operation is not affected by missing external downloads at runtime), (3) deterministic tasks for building, testing, linting/verification, and producing distributable artifacts, and (4) developer and CI workflows that can run without requiring application-runtime network access. Gradle configuration should avoid introducing runtime network dependencies during build or test, and should align with non-functional constraints such as explicit consent for any external network lookups performed by the application itself. | Chosen option: "Gradle", because it is lean and fits our development style. |

*Author on record: Oliver Kopp — [docs/decisions/0003-use-gradle-as-build-tool.md](https://github.com/JabRef/jabref/blob/main/docs/decisions/0003-use-gradle-as-build-tool.md)*



### ADR-0009 — Use Plain JUnit5 for advanced test assertions

| | Sdlicit (generated, as "ADR-0003") | JabRef (real) |
|---|---|---|
| **Alternatives considered** | - Use an external assertion/matcher library (e.g., AssertJ, Hamcrest) for richer fluent assertions.
- Mix JUnit 5 with multiple test utilities/assertion frameworks.
- Rely primarily on snapshot testing frameworks instead of explicit assertions. | - Plain JUnit5
- Hamcrest
- AssertJ |
| **Decision** | Adopt plain JUnit 5 for advanced test assertions across the codebase. Use only JUnit 5 core `org.junit.jupiter.api.Assertions` and JUnit 5 test features to express complex expectations. For readability and reuse, introduce project-local helper assertion methods that wrap JUnit 5 assertions, but do not add additional third-party assertion/matcher libraries.

Standardize on:
- `assertAll(...)` for verifying multiple conditions per test.
- `assertThrows(...)` / `assertDoesNotThrow(...)` for exception and error-path validation.
- `assertTimeout(...)` / `assertTimeoutPreemptively(...)` for performance and termination guarantees where appropriate.
- `@ParameterizedTest` for format/style permutations (e.g., multiple citation styles) and metadata-field variations.
- `DynamicTest` / `@TestFactory` for data-driven assertions when test cases are generated at runtime.
- Keep assertions deterministic by avoiding network calls in tests; use local fixtures/resources.

Rule-of-thumb guidelines:
- Prefer table/parameterized tests over custom matcher dependencies.
- When comparing structured bibliographic outputs (citations/bibliographies), assert on relevant fields/strings explicitly (e.g., citation key, rendered segments) rather than relying on external matcher frameworks.
- Provide small, domain-specific assertion helpers (e.g., `assertCitationKey(...)`, `assertBibEntryField(...)`) that internally call JUnit 5 assertions. | Chosen option: "Plain JUnit5", because comes out best \(see below\).

### Positive Consequences

* Tests are more readable
* More easy to write tests
* More readable assertions

### Negative Consequences

* More complicated testing leads to more complicated assertions |

*Author on record: Oliver Kopp — [docs/decisions/0009-use-plain-junit5-for-testing.md](https://github.com/JabRef/jabref/blob/main/docs/decisions/0009-use-plain-junit5-for-testing.md)*



### ADR-0037 — RAG Architecture Implementation

| | Sdlicit (generated, as "ADR-0004") | JabRef (real) |
|---|---|---|
| **Alternatives considered** | - Remote-only RAG: send queries and/or library data to a hosted retrieval+LLM service
- Local retrieval + remote generation without explicit consent gating
- No RAG; use deterministic rules for metadata enrichment and citation assistance
- RAG only for explanations (no structured metadata suggestions)
- Single retrieval strategy (lexical only or embeddings only) without hybrid retrieval | - Use a hand-crafted RAG
- Use a third-party Java library
- Use a standalone application
- Use an online service |
| **Decision** | Adopt an offline-first RAG architecture with a local indexing/retrieval pipeline and an optional, consent-gated external enrichment path.

Architecture overview:
1) Local retrieval index
- Build and maintain a local retrieval index over the user’s library data (e.g., normalized bibliographic fields such as title/authors/year/publication, plus any locally stored enrichment snapshots).
- Store a “retrieval document” per reference and for any structured metadata/enrichment payload that should be retrievable.
- Create embeddings and/or lexical retrieval structures locally (e.g., local embeddings model and/or BM25/keyword index). Index updates occur on library mutations (add/import/edit/merge).

2) Query understanding and retrieval
- When a user issues a query intended for RAG, transform the query into:
  - A retrieval query (for lexical and/or embedding-based search), and
  - A constrained intent (e.g., “find best matching reference type”, “suggest metadata fields”, “explain citation rationale”).
- Retrieve top-k candidates from the local index.
- Apply reranking locally if needed (e.g., lightweight cross-encoder reranking), keeping the reranking model local.

3) Generation with grounded context
- Provide the generator (LLM) with:
  - The retrieved candidate reference snippets,
  - The specific user task/instruction,
  - Explicit constraints to output structured data (e.g., candidate metadata field suggestions) or citation-ready text.
- Enforce strict output schemas and validation:
  - If generating metadata suggestions, validate against the known metadata model (required fields, allowed additional fields per reference type).
  - If generating explanations/citations, ensure citation keys map to existing references.

4) Consent-gated external enrichment (optional)
- If additional information is needed beyond local retrieval, support an external enrichment adapter, but only after explicit user consent.
- Record:
  - The consent event,
  - The external call request/response metadata,
  - The transformation applied to convert external results into library fields.
- Apply enrichment results in an auditable way (e.g., store an enrichment provenance record per applied field).

5) Offline-first operation and fallback behavior
- Default mode: local-only retrieval and (if available) local generation.
- If the generator or embeddings are not available offline, degrade gracefully:
  - Still support retrieval-based suggestions or deterministic transformations without network calls.
  - Mark generated outputs as unavailable or limited, rather than failing core workflows.

6) Data boundaries and privacy
- Treat library contents as local data.
- If any generation requires a remote model, require an explicit user setting/consent consistent with the external enrichment gating principle, and avoid sending full library contents; send only the minimal retrieved context necessary.

7) Determinism and testability
- Use deterministic settings where possible (seeded generation or template-based generation for structured outputs).
- For tests, use local fixtures and a mock retrieval/generation layer to ensure repeatable RAG behavior.

Implementation deliverables:
- LocalIndexService: indexing, update, and compaction.
- RetrievalService: hybrid retrieval + reranking.
- RAGOrchestrator: retrieval→grounded prompt construction→validated structured output.
- EnrichmentAdapter (optional): consent-gated external enrichment with provenance.
- ProvenanceStore: audit log for enrichment applications.

This design ensures the RAG subsystem enhances workflows without breaking offline operation guarantees and without introducing unauthorized external lookups. | Chosen option: mix of "Use a hand-crafted RAG" and "Use a third-party Java library".

Third-party libraries provide excellent resources for connecting to an LLM or extracting text from PDF files. For RAG,
we mostly used all the machinery provided by `langchain4j`, but there were moments that should be hand-crafted:

* **LLM connection**: due to <https://github.com/langchain4j/langchain4j/issues/1454> (<https://github.com/InAnYan/jabref/issues/77>) this was delegated to another library `jvm-openai`.
* **Embedding generation**: due to <https://github.com/langchain4j/langchain4j/issues/1492> (<ht |

*Author on record: Ruslan — [docs/decisions/0037-rag-architecture-implementation.md](https://github.com/JabRef/jabref/blob/main/docs/decisions/0037-rag-architecture-implementation.md)*



4/4 ADRs generated, 65558 tokens, 153.06s total


## Next steps

- Point this at your **own** project: swap `BRIEF` for your own client brief
  and `topics` for the architectural decisions you actually need to make —
  everything else is unchanged.
- Turn on grounding: set `enable_rag=True` and pass `kb_workdir=` a
  pre-built LightRAG index (see `examples/demo/.sdlicit/knowledge/` for the
  ISO-standards KB shipped with the bundled example project) to ground
  generation in real reference material instead of the brief alone.
- Try the same flow interactively — with Socratic follow-up questions and a
  reviewable artifact tree — via the CLI (`cd cli && python cli_client.py`)
  or the VS Code extension, both driving the same backend this notebook just
  called directly.
- The full replay dataset (`data/replay_dataset.jsonl`) has 50+ real JabRef
  ADRs across tooling, testing, storage, UI, API, CI/CD, and AI/ML — worth
  browsing if you want to try harder or easier topics.